# 01 — EDA dan baseline inventaris

Notebook ini membaca ringkasan bersih `data/cleaned_inventory_summary.csv` (satu baris per mitra dan produk) yang ditulis `src/clean_logs.py`.

File itu adalah snapshot, bukan deret waktu harian. Baseline di bawah memakai rata-rata produk sebagai perkiraan konsumsi mingguan, lalu mengukur WAPE dan MAPE. Lead time pabrik yang dipakai mengikuti pipeline: **6 hari** (rentang operasional 5–7 hari).

`par_level` di sini adalah buffer acuan, bukan kapasitas rak.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LEAD_TIME_DAYS = 6
COVER_SENTINEL = 999.0

CANDIDATES = [
    Path("data/cleaned_inventory_summary.csv"),
    Path("../data/cleaned_inventory_summary.csv"),
    Path("data-science/data/cleaned_inventory_summary.csv"),
]
DATA_PATH = next((path.resolve() for path in CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    searched = ", ".join(str(path) for path in CANDIDATES)
    raise FileNotFoundError(
        "cleaned_inventory_summary.csv tidak ditemukan. "
        f"Jalankan src/clean_logs.py dari data-science. Lokasi yang dicoba: {searched}"
    )

summary = pd.read_csv(DATA_PATH)
summary["is_critical"] = summary["is_critical"].astype(bool)
print(f"Sumber: {DATA_PATH}")
print(f"{len(summary):,} baris, {summary['partner_id'].nunique():,} mitra, {summary['product_name'].nunique():,} produk")
summary.head()

Sumber: C:\daisuki-inventory\data-science\data\cleaned_inventory_summary.csv
424 baris, 106 mitra, 4 produk
Out[0]: 
                             partner_id  ...  is_critical
0  1140f98e-2218-54e2-8709-2245238baa90  ...        False
1  b9d8f9d6-bf42-5b4f-8f59-ce44a481b0eb  ...        False
2  b9d8f9d6-bf42-5b4f-8f59-ce44a481b0eb  ...        False
3  b9d8f9d6-bf42-5b4f-8f59-ce44a481b0eb  ...        False
4  b9d8f9d6-bf42-5b4f-8f59-ce44a481b0eb  ...         True

[5 rows x 12 columns]


## Kualitas snapshot

`days_of_cover = 999` adalah penanda dari pipeline: stok masih ada, tetapi tidak ada konsumsi tercatat pada 7 hari terakhir (`adu = 0`). Angka itu bukan horizon tutup stok yang nyata.

In [2]:
expected = [
    "partner_id",
    "partner_name",
    "product_id",
    "product_name",
    "current_qty",
    "par_level",
    "weekly_consumption",
    "adu",
    "safety_stock",
    "rop",
    "days_of_cover",
    "is_critical",
]
missing_columns = [column for column in expected if column not in summary.columns]
if missing_columns:
    raise ValueError(f"Kolom hilang: {missing_columns}")

observed = summary["weekly_consumption"] > 0
quality = pd.Series(
    {
        "baris": len(summary),
        "nilai_kosong": int(summary.isna().sum().sum()),
        "duplikat_mitra_produk": int(summary.duplicated(["partner_id", "product_id"]).sum()),
        "adu_nol": int((summary["adu"] == 0).sum()),
        "konsumsi_teramati": int(observed.sum()),
        "cover_sentinel_999": int((summary["days_of_cover"] >= COVER_SENTINEL).sum()),
        "kritis": int(summary["is_critical"].sum()),
    }
)
quality

Out[0]: 
baris                    424
nilai_kosong               0
duplikat_mitra_produk      0
adu_nol                  413
konsumsi_teramati         11
cover_sentinel_999       410
kritis                   109
dtype: int64


## Sebaran stok dan konsumsi

Median konsumsi mingguan berada di nol karena sebagian besar mitra belum punya laporan pemakaian pada jendela 7 hari. Rata-rata ditarik oleh sedikit cabang dengan konsumsi sangat tinggi, jadi perbandingan produk memakai median untuk stok dan rata-rata hanya pada baris yang teramati.

In [3]:
numeric = ["current_qty", "par_level", "weekly_consumption", "adu", "safety_stock", "rop", "days_of_cover"]
summary[numeric].describe().round(2)

Out[0]: 
       current_qty  par_level  ...      rop  days_of_cover
count       424.00     424.00  ...   424.00         424.00
mean         34.85      68.30  ...    12.27         966.19
std          26.25      22.83  ...    97.22         177.79
min           0.00      40.00  ...     0.00           0.00
25%          14.00      40.00  ...     0.00         999.00
50%          28.00      60.00  ...     0.00         999.00
75%          51.25      80.00  ...     0.00         999.00
max         150.00     100.00  ...  1025.00         999.00

[8 rows x 7 columns]


In [ ]:
by_product = (
    summary.groupby("product_name", as_index=False)
    .agg(
        mitra=("partner_id", "nunique"),
        median_sisa_stok=("current_qty", "median"),
        rata_rata_sisa_stok=("current_qty", "mean"),
        konsumsi_teramati=("weekly_consumption", lambda values: int((values > 0).sum())),
        median_konsumsi_teramati=(
            "weekly_consumption",
            lambda values: values[values > 0].median() if (values > 0).any() else 0,
        ),
        pangsa_kritis=("is_critical", "mean"),
    )
    .sort_values("pangsa_kritis", ascending=False)
)
by_product["pangsa_kritis"] = (by_product["pangsa_kritis"] * 100).round(1)
for column in ("median_sisa_stok", "rata_rata_sisa_stok", "median_konsumsi_teramati"):
    by_product[column] = by_product[column].round(1)
by_product

Out[0]: 
                 product_name  ...  pangsa_kritis
3         Seblak Jadul Instan  ...           27.4
0          Basreng Daun Jeruk  ...           25.5
2         Makaroni Daun Jeruk  ...           25.5
1  Keripik Pisang Rasa Coklat  ...           24.5

[4 rows x 7 columns]


In [ ]:
partner_health = (
    summary.groupby(["partner_id", "partner_name"], as_index=False)
    .agg(
        produk=("product_id", "nunique"),
        sisa_stok=("current_qty", "sum"),
        produk_kritis=("is_critical", "sum"),
        konsumsi_mingguan=("weekly_consumption", "sum"),
    )
    .sort_values(["produk_kritis", "konsumsi_mingguan"], ascending=False)
)
partner_health.head(10)

Out[0]: 
                              partner_id  ...  konsumsi_mingguan
23  372ce392-cf3c-4119-8b47-ce94e47052d8  ...             3876.0
31  45782e10-f602-5f7c-8479-73eb9af143ee  ...                0.0
76  bf582ad4-b195-4e58-9a20-f40f9890a854  ...                0.0
97  ed67e209-504b-5874-832e-f5436a79a438  ...             1337.0
56  7f8112c6-b6b5-5b34-8770-f5dd5657d66d  ...              616.0
0   003c2623-d955-5f1b-890f-3ce784e077c1  ...                0.0
1   0294b8e1-2bfe-5552-82b6-1553dd33c77e  ...                0.0
2   04b7777d-2d8b-5079-8207-a1d7cba66197  ...                0.0
3   06140801-1db4-57c1-8c29-3d35e1fb5b9f  ...                0.0
4   0650befe-00af-5a8a-87e6-339a2740e4f3  ...                0.0

[10 rows x 6 columns]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

stock = by_product.sort_values("median_sisa_stok")
axes[0].barh(stock["product_name"], stock["median_sisa_stok"], color="#2f6f4e")
axes[0].set_xlabel("Median sisa stok (pcs)")
axes[0].set_title("Sisa stok per produk")

critical = by_product.sort_values("pangsa_kritis")
axes[1].barh(critical["product_name"], critical["pangsa_kritis"], color="#9a3b3b")
axes[1].set_xlabel("Baris kritis (%)")
axes[1].set_title("Pangsa status kritis")

for axis in axes:
    axis.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

: 

## Baseline konsumsi mingguan

Tidak ada histori harian di file ini, jadi moving average dan seasonal naive tidak bisa dilatih dari snapshot. Baseline yang tersedia:

1. **Nol** — perkiraan mayoritas, karena hampir semua baris belum punya konsumsi tercatat.
2. **Rata-rata produk (leave-one-out)** — tiap baris yang konsumsinya teramati dibandingkan dengan rata-rata produk yang sama pada baris teramati lain.

WAPE = jumlah galat absolut / jumlah nilai aktual. MAPE dihitung hanya pada aktual yang lebih dari nol. Sampel teramati kecil, jadi angka ini gambaran kasar, bukan model produksi.

In [ ]:
def wape(actual: np.ndarray, predicted: np.ndarray) -> float:
    denominator = np.abs(actual).sum()
    if denominator == 0:
        return float("nan")
    return float(np.abs(actual - predicted).sum() / denominator)


def mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    mask = actual > 0
    if not mask.any():
        return float("nan")
    return float(np.mean(np.abs(actual[mask] - predicted[mask]) / actual[mask]))


actual_all = summary["weekly_consumption"].to_numpy(dtype=float)
zero_prediction = np.zeros_like(actual_all)
global_mean = np.full_like(actual_all, actual_all.mean())

observed_frame = summary.loc[observed].copy()
loo_prediction = []
for index, row in observed_frame.iterrows():
    peers = observed_frame.loc[
        (observed_frame["product_name"] == row["product_name"]) & (observed_frame.index != index),
        "weekly_consumption",
    ]
    loo_prediction.append(peers.mean() if len(peers) else observed_frame["weekly_consumption"].mean())
loo_prediction = np.asarray(loo_prediction, dtype=float)
actual_observed = observed_frame["weekly_consumption"].to_numpy(dtype=float)

metrics = pd.DataFrame(
    [
        {
            "baseline": "prediksi nol, semua baris",
            "n": len(actual_all),
            "wape": wape(actual_all, zero_prediction),
            "mape": mape(actual_all, zero_prediction),
        },
        {
            "baseline": "rata-rata global, semua baris",
            "n": len(actual_all),
            "wape": wape(actual_all, global_mean),
            "mape": mape(actual_all, global_mean),
        },
        {
            "baseline": "rata-rata produk leave-one-out, baris teramati",
            "n": len(actual_observed),
            "wape": wape(actual_observed, loo_prediction),
            "mape": mape(actual_observed, loo_prediction),
        },
    ]
)
metrics["wape"] = (metrics["wape"] * 100).round(1)
metrics["mape"] = (metrics["mape"] * 100).round(1)
metrics

Out[0]: 
                                         baseline    n   wape   mape
0                       prediksi nol, semua baris  424  100.0  100.0
1                   rata-rata global, semua baris  424  194.8   87.4
2  rata-rata produk leave-one-out, baris teramati   11  112.2  702.7


## Saran pasokan dari buffer yang sudah dihitung

Kolom `adu`, `safety_stock`, dan `rop` berasal dari pipeline (Z = 1,65, lead time 6 hari). Target stok mengikuti rumus baseline di `src/inventory_model.py`: titik pesan ulang ditambah permintaan selama lead time. Kuantitas saran adalah selisih target terhadap sisa stok, dibatasi nol.

Baris tanpa konsumsi teramati mendapat perkiraan permintaan nol. Status kritis pada baris itu datang dari sisa stok yang rendah terhadap buffer, bukan dari ramalan pemakaian.

In [ ]:
lead_demand = summary["adu"] * LEAD_TIME_DAYS
target_stock = np.ceil(summary["rop"] + lead_demand)
recommendations = summary.assign(
    forecast_qty=np.ceil(summary["adu"] * 7).astype(int),
    suggested_qty=np.maximum(0, target_stock - summary["current_qty"]).astype(int),
    days_of_cover=summary["days_of_cover"].where(summary["days_of_cover"] < COVER_SENTINEL),
    model_version="baseline-snapshot-adu-v1",
)

actionable = recommendations[recommendations["suggested_qty"] > 0].sort_values(
    "suggested_qty", ascending=False
)
print(f"Baris dengan saran pasokan > 0: {len(actionable):,} dari {len(recommendations):,}")
actionable[
    [
        "partner_name",
        "product_name",
        "current_qty",
        "forecast_qty",
        "suggested_qty",
        "days_of_cover",
        "is_critical",
    ]
].head(15)

Baris dengan saran pasokan > 0: 9 dari 424
Out[0]: 
                 partner_name  ...  is_critical
421            kopken cijeruk  ...         True
54             kopken cijeruk  ...         True
420  Restoran Purnama Cirebon  ...         True
57             kopken cijeruk  ...         True
39             kopken cijeruk  ...         True
199     Restoran Mentari Solo  ...         True
198  Restoran Purnama Cirebon  ...        False
422     Restoran Mentari Solo  ...        False
419  Restoran Purnama Cirebon  ...        False

[9 rows x 7 columns]
